In [2]:
# -*- coding: utf-8 -*-
# pylint: disable=C0301

"""An agent class that implements the CodeAct agent.
This agent can execute code interactively as actions.
More details can be found at the paper of codeactagent
https://arxiv.org/abs/2402.01030
and the original repo of codeact https://github.com/xingyaoww/code-act
"""

from agentscope.agents import AgentBase
from agentscope.message import Msg
from agentscope.service import (
    ServiceResponse,
    ServiceExecStatus,
    NoteBookExecutor
)
from agentscope.parsers import RegexTaggedContentParser

SYSTEM_MESSAGE = """
You are a helpful assistant that gives helpful, detailed, and polite answers to the user's questions.
You should interact with the interactive Python (Jupyter Notebook) environment and receive the
corresponding output when needed. The code written by assistaant should be enclosed using <execute>
tag, for example:<execute> print('Hello World!') </execute>
You should attempt fewer things at a time instead of putting too much code in one <execute> block.
You can install packages through PIP by <execute> !pip install [package needed] </execute> and should
always import packages and define variables before startting to use them.
You should stop <execute> and provide an answer when they have already obtained the answer from the
execution result. Whenever possible, execute the code for the user using (execute> instead of
providing it.
Your response should be concise, but do express their thoughts. Always write the code in <execute>
block to execute them.
You should not ask for the user's input unless necessary. Solve the task on your own and leave no
unanswered questions behind.
You should do every thing by yourself.
""" # noqa

EXAMPLE_MESSAGE = """
Additionally, you are provided with the following code available:
{example_code}
The above code is already available in your interactive Python (Jupyter Notebook) environment,
allowing you to directly use these variables and fundctions without needing to redeclare them
""" # noqa



In [3]:
class CodeActAgent (AgentBase):
    """
    The implementation of CodeAct-agent.
    The agent can execute code interactively as actions
    More details can be found at the paper of codeact ageent
    https://arxiv.org/abs/2402.01030
    and the original repo of codeact https://github.com/xingyaoww/code-act
    """
    def __init__(
        self,
        name: str,
        model_config_name: str,
        example_code: str = "",
        )->None:
        """
        Initialize the CodeActAgent.
        Args:
            name(`str`):
                The name of the agent.
            model_config_name(`str`):
                The name of the model configuration.
            example_code (Optional `str`):
                The example code to be executed before the interaction.
                You can import reference libs, define variables
                and functions to be called. For example:
                
            \```python
            from agentscope.service import bing_search
            import os

            api_key = "{YOUR_BING_API_KEY}"

            def search(question: str):
                return bing_search(question, api_key=api_key, num_results=3).content
        """ # noqa

        super().__init__(name=name, model_config_name=model_config_name)
        self.n_max_executions = 5
        self.example_code = example_code
        self.code_executor = NoteBookExecutor()

        sys_msg = Msg(name="system", role="system", content=SYSTEM_MESSAGE)
        example_msg = Msg(
            name="user",
            role="user",
            content=EXAMPLE_MESSAGE.format(example_code=self.example_code),
        )

        self.memory.add(sys_msg)

        if self.example_code != "":
            code_execution_result = self.code_executor.run_code_on_notebook(
                self.example_code,
            )
            code_exec_msg = self.handle_code_result(
                code_execution_result,
                "Example Code excuted: ",
            )
            self.memory.add(example_msg)
            self.memory.add(code_exec_msg)
            self.speak(code_exec_msg)

        self.parser = RegexTaggedContentParser(try_parse_json=False)

    def handle_code_result(
            self, 
            code_execution_result: ServiceResponse, 
            content_pre_sring: str = "",
        ) -> Msg:
        """
        返回代码结果的消息
        """
        code_exec_content = content_pre_sring

        if code_execution_result.status == ServiceExecStatus.SUCCESS:
            code_exec_content += "执行成功:\n"
        else:
            code_exec_content += "执行失败:\n"
        code_exec_content += "执行输出:\n" + str(
                code_execution_result.content,
            )

        return Msg(name="user", role="user", content=code_exec_content)


    def reply(self, x: Msg = None) -> Msg:
        """
        实现CodeAct代理的回复函数。
        """

        self.memory.add(x)

        execution_count = 0
        while (
            self.memory.get_memory(1)[-1].role == "user"
            and execution_count < self.n_max_executions
        ):
            prompt = self.model.format(self.memory.get_memory())
            model_res = self.model(prompt)
            msg_res = Msg(
                name=self.name,
                content=model_res.text,
                role="assistant",
            )

            self.memory.add(msg_res)
            self.speak(msg_res)

            res = self.parser.parse(model_res)
            code = res.parsed.get("execute")

            if code is not None:
                code = code.strip()
                code_execution_result = (
                    self.code_executor.run_code_on_notebook(code)
                )

            execution_count += 1
            code_exec_msg = self.handle_code_result(code_execution_result)
            self.memory.add(code_exec_msg)
            self.speak(code_exec_msg)
        
        if execution_count == self.n_max_executions:
            assert self.memory.get_memory(1)[-1].role == "user"
            code_max_exec_msg = Msg(
                name="assistant",
                role="assistant",
                content=(
                    f"I have reached the maximum number of executions ({self.n_max_executions}). "
                    "Can you assist me or ask me another question?"
                ),
            )
            self.memory.add(code_max_exec_msg)
            self.speak(code_max_exec_msg)
            return code_max_exec_msg

        return msg_res

In [5]:
# !pip install notebook nbclient nbformat

  Using cached jsonschema-4.23.0-py3-none-any.whl.metadata (7.9 kB)
  Using cached jsonschema_specifications-2023.12.1-py3-none-any.whl.metadata (3.0 kB)
  Using cached referencing-0.35.1-py3-none-any.whl.metadata (2.8 kB)
  Using cached rpds_py-0.20.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (4.2 kB)
  Using cached PyYAML-6.0.2-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (2.1 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.0/5.0 MB 5.8 MB/s eta 0:00:00a 0:00:01
Using cached jsonschema-4.23.0-py3-none-any.whl (88 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 15.6 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.6/9.6 MB 14.8 MB/s eta 0:00:0000:0100:01
Using cached jsonschema_specifications-2023.12.1-py3-none-any.whl (18 kB)
Using cached referencing-0.35.1-py3-none-any.whl (26 kB)
Using cached rpds_py-0.20.0-cp310-cp310-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (354 kB)
Using cached PyYAML-6.0

In [4]:
import nest_asyncio
import agentscope

modelscope_cfg_dict = "dashscope/model_config.json"


agentscope.init(
    model_configs=modelscope_cfg_dict,
    project="CodeAct",
    save_api_invoke=True,
)

nest_asyncio.apply()
agent = CodeActAgent(
    name="assistant",
    model_config_name="my_dashscope_chat_config"
)


2024-09-29 20:25:28.341 | INFO     | agentscope.manager._model:load_model_configs:115 - Load configs for model wrapper: my_dashscope_chat_config
2024-09-29 20:25:28.361 | INFO     | agentscope.models.model:__init__:203 - Initialize model by configuration [my_dashscope_chat_config]


In [5]:
from loguru import logger
from agentscope.message import Msg

mss = Msg(
    name="user",
    content="Given y = 0.9x + 6.1, randomly sample data points as pairs of (x, y). Then fit a linear regression on the sampled data and plot the points, fitted line, and ground-truth line.",
    role="user"
)

logger.chat(mss) 
answer_mssl = agent(mss)

user: Given y = 0.9x + 6.1, randomly sample data points as pairs of (x, y). Then fit a linear regression on the sampled data and plot the points, fitted line, and ground-truth line.
assistant: <execute>
import numpy as np
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Generate random data points
np.random.seed(0)
x = np.random.rand(50) * 10
y = 0.9 * x + 6.1 + np.random.randn(50) * 0.5

# Fit a linear regression model
model = LinearRegression()
model.fit(x[:, np.newaxis], y)

# Plot the data points, fitted line, and ground-truth line
plt.figure(figsize=(12, 6))

# Plot the data points
plt.scatter(x, y, color='blue', label='Sampled Data Points')

# Plot the fitted line
x_fit = np.linspace(0, 10, 100)
y_fit = model.predict(x_fit[:, np.newaxis])
plt.plot(x_fit, y_fit, color='red', label='Fitted Line')

# Plot the ground-truth line
y_true = 0.9 * x_fit + 6.1
plt.plot(x_fit, y_true, color='green', label='Ground-Truth Line')

# Add labels and legend
plt.